# Exp06 `scl_li` cell — extend to 3 seeds (hardening)

Exp06's 8-cell ablation ran single-seed (42) — `scl_li` was the best cell on all 3 raw metrics simultaneously (cls 0.7807 / span_em 0.6415 / span_f1 0.8332). Per the 2026-06-23 council hardening list, 3-seeding `scl_li` specifically (not all 8 cells — too expensive) de-risks the "best cell" claim against single-seed noise.

**Why this notebook calls the trainer directly instead of `run_06b_idiombert_v2.sh`:** that wrapper hardcodes `SEED=42` (not env-overridable like `SEEDS` in run_15/16b/17b) — by design, since Exp06 was scoped single-seed. This notebook reimplements the wrapper's persistence gate + skip/resume logic inline for just the 2 new seeds (**123, 7**), using the exact same `lr`/`batch`/`langs`/flags the wrapper uses for `scl_li` (verified against `run_06b_idiombert_v2.sh` lines ~88-96, 2026-06-23: `lr=2e-5, batch=32, langs=[English,Spanish,Hindi,Telugu], test_langs=[...,Indonesian], flags=--use_scl --use_li`).

**Persistence:** each seed's output dir is symlinked to Drive and gated before training — hard-fails if not Drive-backed. If the session times out, **just re-run the training cell** — a seed whose `test_predictions.jsonl` already exists is skipped.

Run cells top to bottom. Use a **GPU** runtime (Runtime → Change runtime type → T4/A100).

In [ ]:
# 1. Config
REPO_URL = 'https://github.com/JustLetMeBeHello/Idiomator_Research.git'
BRANCH   = 'main'
REPO     = '/content/Idiomator_Research'   # absolute — never use a relative %cd
SEEDS    = ['123', '7']                      # the 2 NEW seeds — 42 already done+registered as models/idiombert_v2/scl_li
FORCE    = False
DRIVE_OUT = '/content/drive/MyDrive/IdiomatorRigor'
# values below MUST match run_06b_idiombert_v2.sh's scl_li cell exactly — re-grep
# the script before running if it has changed since 2026-06-23
LR = '2e-5'
BATCH = '32'
LANGS = ['English', 'Spanish', 'Hindi', 'Telugu']
TEST_LANGS = ['English', 'Spanish', 'Hindi', 'Telugu', 'Indonesian']
FLAGS = ['--use_scl', '--use_li']
print('repo:', REPO, '| new seeds:', SEEDS)

In [ ]:
# 2. Clone / refresh repo, pin absolute cwd, kill any nested duplicate clone
import os, subprocess, sys, shutil
from pathlib import Path
nested = os.path.join(REPO, 'Idiomator_Research')
if os.path.isdir(nested):
    subprocess.run(['rm', '-rf', nested], check=True)
if os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '--quiet', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '--quiet', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--quiet', '--branch', BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)   # ABSOLUTE
print('cwd:', os.getcwd())
print('head:', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())
TRAINER = 'experiments/rigor/run_06_idiombert_v2.py'
assert Path(TRAINER).exists(), 'trainer missing — wrong repo/branch?'
# sanity-check the wrapper's scl_li flags still match this notebook's hardcoded copy
wrapper = Path('experiments/rigor/run_06b_idiombert_v2.sh').read_text()
assert 'scl_li)      echo "--use_scl --use_li"' in wrapper, 'scl_li flags in run_06b changed — update this notebook before running'
assert 'LR=2e-5' in wrapper and 'BATCH=32' in wrapper, 'lr/batch in run_06b changed — update this notebook before running'

In [ ]:
# 3. Install deps + confirm GPU
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Requirements.txt'], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU — switch runtime to T4/A100'

In [ ]:
# 4. Mount Drive
from google.colab import drive
drive.mount('/content/drive')
Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
print('DRIVE_OUT =', DRIVE_OUT)

In [ ]:
# 5. GPU smoke test (~1-2 min): 1 epoch, English only, throwaway dir, scl_li flags
!python {TRAINER} --output_dir /tmp/_scl_li_smoke --langs English --test_langs English \
    --epochs 1 --batch_size 8 --use_scl --use_li

In [ ]:
# 6. Per-seed persistence gate (same hard rule as run_06b's gate_dir, reimplemented
#    here since this notebook runs the trainer directly, not through the wrapper)
def gate_dir(out_dir: str, drive_out_root: str):
    name = out_dir.replace('models/', '', 1)
    drive_target = os.path.join(drive_out_root, name)
    os.makedirs(drive_target, exist_ok=True)
    if os.path.exists(out_dir) or os.path.islink(out_dir):
        if os.path.islink(out_dir):
            os.remove(out_dir)
        else:
            shutil.rmtree(out_dir)
    os.makedirs(os.path.dirname(out_dir), exist_ok=True)
    os.symlink(drive_target, out_dir)
    assert os.path.islink(out_dir), f'{out_dir} is not a symlink — writes would be ephemeral'
    real = os.path.realpath(out_dir)
    assert real.startswith(os.path.realpath(drive_out_root)), f'{out_dir} -> {real} not under Drive'
    probe = os.path.join(out_dir, '.persist_probe')
    open(probe, 'w').write('ok')
    assert open(probe).read() == 'ok'
    os.remove(probe)
    print(f'  \u2713 persistence gate passed \u2014 {out_dir} is Drive-backed and writable')

def should_run(out_dir: str, force: bool) -> bool:
    preds = os.path.join(out_dir, 'test_predictions.jsonl')
    if not force and os.path.exists(preds):
        print(f'  \u2713 skip \u2014 {preds} exists (set FORCE=True to retrain)')
        return False
    return True

In [ ]:
# 7. FULL RUN \u2014 seeds 123 and 7, scl_li cell, Drive-gated, resumable.
#    Re-run this cell after any timeout: a seed whose preds already exist is skipped.
for seed in SEEDS:
    out_dir = f'models/idiombert_v2_s{seed}/scl_li'
    print(f'\n\u2500\u2500 scl_li seed={seed} \u2192 {out_dir} \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
    if not should_run(out_dir, FORCE):
        continue
    gate_dir(out_dir, DRIVE_OUT)
    cmd = [sys.executable, TRAINER,
           '--output_dir', out_dir,
           '--model_name', 'bert-base-multilingual-cased',
           '--langs', *LANGS,
           '--test_langs', *TEST_LANGS,
           '--lr', LR, '--batch_size', BATCH, '--seed', seed,
           *FLAGS]
    print('  $', ' '.join(cmd))
    log_path = f'{DRIVE_OUT}/exp06_scl_li_s{seed}_console.log'
    with open(log_path, 'a') as logf:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
            logf.write(line)
        proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'seed {seed} failed, see {log_path}')
    if os.path.exists(os.path.join(out_dir, 'test_predictions.jsonl')):
        print(f'  \u2713 seed {seed} done')

In [ ]:
# 8. Persistence readback \u2014 reads metrics.json back FROM DRIVE for both new seeds
import json
print(f"{'seed':<6}{'test_cls_f1':<14}{'test_exact':<12}{'test_ovlp_f1':<14}{'persisted?'}")
for seed in SEEDS:
    mp = Path(DRIVE_OUT) / f'idiombert_v2_s{seed}' / 'scl_li' / 'metrics.json'
    if not mp.exists():
        print(f'{seed:<6}\u2014 metrics.json NOT on Drive (incomplete / not persisted)')
        continue
    M = json.load(open(mp))
    print(f"{seed:<6}{M.get('test_cls_macro_f1', M.get('cls_f1')):<14}{M.get('test_span_exact', M.get('span_em')):<12}{M.get('test_span_overlap', M.get('span_f1')):<14}yes")
print('\nNext (local, not in this notebook): pull both seeds, register via')
print('Full_evaluation.py --joint_preds against models/idiombert_v2/scl_li (seed 42, already')
print('registered), compute 3-seed mean\u00b1sd, check_metric_drift.py.')